# Circular Text Fingerprints vs MinHash: Near-Duplicate Detection

This notebook demonstrates comparing two methods for detecting near-duplicate question pairs:
- **MinHash (baseline)**: Standard locality-sensitive hashing using 3-gram shingling
- **Circular Text Fingerprints (CTF)**: Adapts ECFP molecular fingerprints to token sequences with neighborhood propagation

We evaluate both methods on the Quora Question Pairs (QQP) dataset, measuring precision, recall, and F1 score across text length buckets.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')
_pip('datasets==4.0.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'pandas==2.2.2')

In [ ]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
from typing import Any

import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2cec91-circular-text-fingerprints-evaluating-ec/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL, with fallback to local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
demo_data = load_data()
print(f"Loaded demo data with {len(demo_data['datasets'][0]['examples'])} examples")
print(f"Method: {demo_data['metadata']['method_name']}")

## Configuration

Below are the tunable parameters for the experiment. For the demo notebook, we use minimal values that complete quickly while still producing meaningful results. Scale these up (e.g., N_PAIRS to 100, 500, 1000, 10000) to reproduce the full experiment.

In [ ]:
# ── Tunable parameters (start with MINIMUM for demo) ────────────────────────
N_PAIRS = 50  # Demo: 50 pairs (25 dup, 25 non-dup). Full: 10000
N_HASHES = 8  # Demo: 8 hashes. Full: 128
SEED = 42

# Keep these fixed (from original experiment)
SHINGLE_K = 3
CTF_RADII = [1, 2, 3]
CTF_WINDOW = 1

print(f"Demo config: N_PAIRS={N_PAIRS}, N_HASHES={N_HASHES}, SEED={SEED}")

## MinHash Baseline

MinHash is a locality-sensitive hashing technique that estimates Jaccard similarity between sets.
Here we apply it to token n-grams (shingles) from each question pair.

In [ ]:
def _stable_hash(val: Any) -> int:
    """Stable, seed-independent integer hash via SHA-256."""
    h = hashlib.sha256(str(val).encode("utf-8")).digest()
    return int.from_bytes(h[:8], "little")

def _multi_hash(val: Any, seed: int) -> int:
    """Hash with seed for MinHash."""
    h = hashlib.sha256(f"{seed}:{val}".encode("utf-8")).digest()
    return int.from_bytes(h[:8], "little")

def make_shingles(text: str, k: int = SHINGLE_K) -> set:
    """Split text into k-gram shingles (token-based)."""
    tokens = text.lower().split()
    if len(tokens) < k:
        return set(tuple(tokens))  # fallback to unigrams if too short
    return {tuple(tokens[i : i + k]) for i in range(len(tokens) - k + 1)}

def minhash_signature(shingles: set, num_hashes: int = N_HASHES) -> np.ndarray:
    """Compute MinHash signature for a set of shingles."""
    sig = np.full(num_hashes, np.iinfo(np.uint64).max, dtype=np.uint64)
    for shingle in shingles:
        key = str(shingle)
        for seed in range(num_hashes):
            h = _multi_hash(key, seed) & 0xFFFFFFFFFFFFFFFF
            if h < sig[seed]:
                sig[seed] = h
    return sig

def jaccard_minhash(sig1: np.ndarray, sig2: np.ndarray) -> float:
    """Estimate Jaccard similarity from MinHash signatures."""
    return float(np.mean(sig1 == sig2))

print("MinHash functions defined.")

## Circular Text Fingerprints (CTF)

CTF adapts ECFP (Extended Connectivity Fingerprints) from chemistry to text. Each token starts with its own hash,
then iteratively accumulates neighborhood hashes across R iterations. The resulting feature set captures structural
token context that n-gram shingling may miss.

In [ ]:
def ctf_fingerprint(text: str, radius: int = 2) -> frozenset:
    """Compute Circular Text Fingerprint with iterative neighborhood hashing."""
    tokens = text.lower().split()
    if not tokens:
        return frozenset()
    n = len(tokens)
    current = [_stable_hash(t) for t in tokens]
    features: set = set(current)

    for _ in range(radius):
        nxt = []
        for i in range(n):
            # Gather neighbor hashes (within window=1 distance)
            neighbors = sorted(
                current[j] for j in range(max(0, i - CTF_WINDOW), min(n, i + CTF_WINDOW + 1)) if j != i
            )
            # Hash the token + its neighbors as a new feature
            new_feat = _stable_hash((current[i], tuple(neighbors))) & 0xFFFFFFFFFFFFFFFF
            nxt.append(new_feat)
            features.add(new_feat)
        current = nxt

    return frozenset(features)

def jaccard_sets(a: frozenset, b: frozenset) -> float:
    """Exact Jaccard similarity between two feature sets."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union

print("CTF functions defined.")

## Data Preparation & Processing

This section loads synthetic balanced question pairs, computes similarity scores using both methods,
optimizes decision thresholds on train+val split, and evaluates on the test set.

In [ ]:
# Create synthetic balanced pairs for demo (in full experiment, these come from QQP dataset)
print(f"Preparing {N_PAIRS} balanced question pairs…")
rng = np.random.default_rng(SEED)

# Generate simple synthetic pairs: duplicate pairs (label=1) and non-duplicate pairs (label=0)
duplicate_questions = [
    ("What is machine learning", "What is machine learning"),
    ("How do I learn Python", "How to learn Python programming"),
    ("What are neural networks", "What are artificial neural networks"),
    ("How does deep learning work", "How does deep learning function"),
    ("What is natural language processing", "What is NLP"),
]

non_duplicate_questions = [
    ("What is machine learning", "How do I cook pasta"),
    ("How do I learn Python", "What is the capital of France"),
    ("What are neural networks", "How to fix a bicycle"),
    ("How does deep learning work", "What is basketball"),
    ("What is natural language processing", "Where is the nearest coffee shop"),
]

# Repeat and shuffle to reach N_PAIRS
pairs = []
dup_count = N_PAIRS // 2
non_dup_count = N_PAIRS - dup_count

for _ in range((dup_count // len(duplicate_questions)) + 1):
    pairs.extend([(q1, q2, 1) for q1, q2 in duplicate_questions])
for _ in range((non_dup_count // len(non_duplicate_questions)) + 1):
    pairs.extend([(q1, q2, 0) for q1, q2 in non_duplicate_questions])

pairs = pairs[:N_PAIRS]
rng.shuffle(pairs)

sample = [{"q1": q1, "q2": q2, "label": label} for q1, q2, label in pairs]
labels = np.array([p["label"] for p in sample])
lengths = np.array([len(p["q1"]) + len(p["q2"]) for p in sample])

# Split: 60/20/20
idx = np.arange(len(sample))
train_idx, tmp_idx = train_test_split(idx, test_size=0.4, random_state=SEED, stratify=labels)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.5, random_state=SEED, stratify=labels[tmp_idx])
train_val_idx = np.concatenate([train_idx, val_idx])

print(f"Split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
print(f"Labels: {np.sum(labels)} duplicates, {len(labels) - np.sum(labels)} non-duplicates")

## Compute Similarity Scores

Compute MinHash and CTF similarity scores for all question pairs.

In [ ]:
# ── MinHash scores ────────────────────────────────────────────────────────
print("Computing MinHash signatures…")
mh_scores = np.zeros(len(sample))
for i, p in enumerate(sample):
    s1 = make_shingles(p["q1"])
    s2 = make_shingles(p["q2"])
    if not s1 and not s2:
        mh_scores[i] = 1.0
    elif not s1 or not s2:
        mh_scores[i] = 0.0
    else:
        sig1 = minhash_signature(s1)
        sig2 = minhash_signature(s2)
        mh_scores[i] = jaccard_minhash(sig1, sig2)
print(f"MinHash done. Mean score: {np.mean(mh_scores):.4f}")

# ── CTF scores ───────────────────────────────────────────────────────────
ctf_scores: dict = {}
for radius in CTF_RADII:
    print(f"Computing CTF fingerprints (radius={radius})…")
    scores = np.zeros(len(sample))
    for i, p in enumerate(sample):
        fp1 = ctf_fingerprint(p["q1"], radius=radius)
        fp2 = ctf_fingerprint(p["q2"], radius=radius)
        scores[i] = jaccard_sets(fp1, fp2)
    ctf_scores[radius] = scores
    print(f"CTF R={radius} done. Mean score: {np.mean(scores):.4f}")

## Threshold Optimization

Sweep thresholds on train+val to maximize F1 score, then evaluate on test set.

In [ ]:
def sweep_threshold(scores: np.ndarray, labels: np.ndarray) -> tuple:
    """Find best threshold that maximizes F1 on given scores and labels."""
    best_f1, best_thr = 0.0, 0.5
    for thr in np.arange(0.0, 1.01, 0.05):  # coarser sweep for demo
        preds = (scores >= thr).astype(int)
        tp = int(np.sum((preds == 1) & (labels == 1)))
        fp = int(np.sum((preds == 1) & (labels == 0)))
        fn = int(np.sum((preds == 0) & (labels == 1)))
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr, best_f1

print("Optimizing thresholds on train+val…")
tv_labels = labels[train_val_idx]

mh_thr, mh_tv_f1 = sweep_threshold(mh_scores[train_val_idx], tv_labels)
print(f"MinHash: best_threshold={mh_thr:.2f}, train+val F1={mh_tv_f1:.4f}")

ctf_thresholds: dict = {}
for radius in CTF_RADII:
    thr, tv_f1 = sweep_threshold(ctf_scores[radius][train_val_idx], tv_labels)
    ctf_thresholds[radius] = thr
    print(f"CTF R={radius}: best_threshold={thr:.2f}, train+val F1={tv_f1:.4f}")

## Test Evaluation

Evaluate both methods on the held-out test set using the optimized thresholds.

In [ ]:
def evaluate(scores: np.ndarray, labels: np.ndarray, thr: float) -> dict:
    """Compute precision, recall, F1 for a given threshold."""
    preds = (scores >= thr).astype(int)
    tp = int(np.sum((preds == 1) & (labels == 1)))
    fp = int(np.sum((preds == 1) & (labels == 0)))
    tn = int(np.sum((preds == 0) & (labels == 0)))
    fn = int(np.sum((preds == 0) & (labels == 1)))
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn, "count": len(labels)}

def stratify_evaluate(
    scores: np.ndarray, labels: np.ndarray, lengths: np.ndarray, thr: float
) -> dict:
    """Evaluate with stratification by text length."""
    short_mask = lengths < 40
    mid_mask = (lengths >= 40) & (lengths <= 100)
    long_mask = lengths > 100
    result = {"overall": evaluate(scores, labels, thr)}
    for name, mask in [("short", short_mask), ("medium", mid_mask), ("long", long_mask)]:
        if mask.sum() > 0:
            result[name] = evaluate(scores[mask], labels[mask], thr)
        else:
            result[name] = {"count": 0, "precision": 0.0, "recall": 0.0, "f1": 0.0}
    return result

print("Evaluating on test set…")
test_labels = labels[test_idx]
test_lengths = lengths[test_idx]

mh_results = stratify_evaluate(mh_scores[test_idx], test_labels, test_lengths, mh_thr)
ctf_results: dict = {}
for radius in CTF_RADII:
    ctf_results[f"radius_{radius}"] = stratify_evaluate(
        ctf_scores[radius][test_idx], test_labels, test_lengths, ctf_thresholds[radius]
    )

print("Evaluation done.")

## Results & Visualization

Compare MinHash baseline against CTF at multiple radii. Display results in table and chart formats.

In [ ]:
# ── Success criterion ────────────────────────────────────────────────────
mh_f1 = mh_results["overall"]["f1"]
ctf_best_f1 = max(ctf_results[f"radius_{r}"]["overall"]["f1"] for r in CTF_RADII)
ctf_best_r = max(CTF_RADII, key=lambda r: ctf_results[f"radius_{r}"]["overall"]["f1"])
delta = ctf_best_f1 - mh_f1

print(f"\n=== SUMMARY ===")
print(f"MinHash F1: {mh_f1:.4f}")
print(f"CTF Best F1 (R={ctf_best_r}): {ctf_best_f1:.4f}")
print(f"Improvement: {delta:.4f} F1 points")

# ── Results table ────────────────────────────────────────────────────────
print(f"\n=== RESULTS TABLE ===")
print(f"{'Method':<14} {'R':>2} {'Thr':>5} {'Overall F1':>10} {'Short F1':>9} {'Med F1':>7} {'Long F1':>8}")
print("-" * 70)
print(f"{'MinHash':<14} {'N/A':>2} {mh_thr:>5.2f} {mh_f1:>10.4f} {mh_results.get('short', {}).get('f1', 0):>9.4f} {mh_results.get('medium', {}).get('f1', 0):>7.4f} {mh_results.get('long', {}).get('f1', 0):>8.4f}")
for r in CTF_RADII:
    cr = ctf_results[f"radius_{r}"]
    print(f"{'CTF':<14} {r:>2} {ctf_thresholds[r]:>5.2f} {cr['overall']['f1']:>10.4f} {cr.get('short', {}).get('f1', 0):>9.4f} {cr.get('medium', {}).get('f1', 0):>7.4f} {cr.get('long', {}).get('f1', 0):>8.4f}")

In [ ]:
# ── Visualization ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall F1 comparison
methods = ["MinHash"] + [f"CTF\n(R={r})" for r in CTF_RADII]
f1_scores = [mh_f1] + [ctf_results[f"radius_{r}"]["overall"]["f1"] for r in CTF_RADII]
colors = ["steelblue"] + ["coral", "lightgreen", "gold"]

axes[0].bar(methods, f1_scores, color=colors, alpha=0.7, edgecolor="black")
axes[0].set_ylabel("F1 Score", fontsize=11)
axes[0].set_title("Overall F1 Comparison (Test Set)", fontsize=12, fontweight="bold")
axes[0].set_ylim([0, 1.0])
for i, v in enumerate(f1_scores):
    axes[0].text(i, v + 0.02, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# F1 by text length bucket
buckets = ["Short\n(<40 chars)", "Medium\n(40-100)", "Long\n(>100)"]
bucket_keys = ["short", "medium", "long"]
x = np.arange(len(buckets))
width = 0.2

mh_f1_by_len = [mh_results.get(k, {}).get("f1", 0) for k in bucket_keys]
axes[1].bar(x - width, mh_f1_by_len, width, label="MinHash", color="steelblue", alpha=0.7)
for i, r in enumerate(CTF_RADII):
    ctf_f1_by_len = [ctf_results[f"radius_{r}"].get(k, {}).get("f1", 0) for k in bucket_keys]
    axes[1].bar(x + (i-1)*width, ctf_f1_by_len, width, label=f"CTF (R={r})", alpha=0.7)

axes[1].set_ylabel("F1 Score", fontsize=11)
axes[1].set_title("F1 by Text Length (Test Set)", fontsize=12, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(buckets)
axes[1].legend(loc="lower right", fontsize=9)
axes[1].set_ylim([0, 1.0])
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete.")